# Chapter 2: Geometry and Nearest Neighbors

> Every example is just a point in space — and once you accept that, distance becomes a learning algorithm.

**Type:** Learn + Build &nbsp;|&nbsp; **Language:** Python &nbsp;|&nbsp; **Prerequisites:** Chapter 1 (Decision Trees) &nbsp;|&nbsp; **Time:** ~45 minutes
**Source:** *A Course in Machine Learning*, Hal Daumé III — Chapter 2

---

## Learning Objectives

- Describe a dataset as a collection of points in a high-dimensional feature space
- Implement a **K-Nearest Neighbors (KNN)** classifier from scratch and understand its decision rule
- Implement **K-Means** clustering from scratch and understand the assignment/update loop
- Explain how the hyperparameter **K** trades off underfitting and overfitting for KNN
- Explain the **curse of dimensionality** and why distances become uninformative in high dimensions

## The Problem

Decision trees (Chapter 1) work by asking a small sequence of yes/no questions about the most informative features. But what if you don't want to hand-pick which features matter?

A very different idea is: represent every example as a vector of numbers (a point in space), and classify a new point based on which *other* points it is closest to. This is the **geometric view of data**, and it leads naturally to:

- **K-Nearest Neighbors (KNN)** for classification
- **K-Means** for unsupervised clustering

## The Concept

**KNN's decision process:**

```
New test point
      │
      ▼
Compute distance to every training point
      │
      ▼
Sort by distance
      │
      ▼
Take K closest neighbors
      │
      ▼
Majority vote of their labels
      │
      ▼
Predicted label
```

### Key Ideas

- **No training phase:** KNN just stores the training data; all the work happens at prediction time.
- **K controls the bias/variance trade-off:** K=1 memorizes noise (overfits); a very large K collapses toward always predicting the majority class (underfits).
- **Feature scale matters:** since KNN relies purely on distance, unscaled features (e.g., one measured in millimeters, another in kilometers) silently dominate the distance calculation.
- **K-Means is the unsupervised cousin:** instead of using known labels, it alternates between assigning points to the nearest of K cluster centers and recomputing those centers as the mean of their assigned points.

## Build It

### Setup

We'll use NumPy for the from-scratch math, and several scikit-learn pieces purely for real datasets, reference implementations to validate against, and evaluation metrics (`accuracy_score`, `adjusted_rand_score`).

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, adjusted_rand_score

RNG = np.random.RandomState(42)

### Step 1: K-Nearest Neighbors, From Scratch (Algorithm 3 in the book)

KNN has an unusual structure compared to most learning algorithms: `fit` does almost nothing — it just stores the training data. All the real work happens in `predict`:

1. For a test point, compute its Euclidean distance to every stored training point
2. Sort training points by distance and keep the closest `k`
3. Return the majority label among those `k` neighbors

In [2]:
class KNNFromScratch:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.asarray(X)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X)
        preds = np.empty(X.shape[0], dtype=self.y_train.dtype)
        for i, x in enumerate(X):
            dists = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
            nn_idx = np.argsort(dists)[: self.k]
            nn_labels = self.y_train[nn_idx]
            values, counts = np.unique(nn_labels, return_counts=True)
            preds[i] = values[np.argmax(counts)]
        return preds

### Step 2: K-Means, From Scratch (Algorithm 4 in the book)

K-Means alternates between two steps until the cluster centers stop moving:

1. **Assignment step:** assign every point to its nearest of the `k` current centers
2. **Update step:** recompute each center as the mean of the points assigned to it

Initialization here picks `k` random distinct training points as the starting centers.

In [3]:
class KMeansFromScratch:
    def __init__(self, k=3, max_iter=100, random_state=0):
        self.k = k
        self.max_iter = max_iter
        self.random_state = random_state

    def fit(self, X):
        X = np.asarray(X)
        rng = np.random.RandomState(self.random_state)
        init_idx = rng.choice(X.shape[0], self.k, replace=False)
        self.centers_ = X[init_idx].copy()

        for _ in range(self.max_iter):
            dists = np.linalg.norm(X[:, None, :] - self.centers_[None, :, :], axis=2)
            labels = np.argmin(dists, axis=1)

            new_centers = np.empty_like(self.centers_)
            for k in range(self.k):
                members = X[labels == k]
                if len(members) > 0:
                    new_centers[k] = members.mean(axis=0)
                else:
                    new_centers[k] = self.centers_[k]

            if np.allclose(new_centers, self.centers_):
                self.centers_ = new_centers
                break
            self.centers_ = new_centers

        self.labels_ = labels
        return self

    def predict(self, X):
        X = np.asarray(X)
        dists = np.linalg.norm(X[:, None, :] - self.centers_[None, :, :], axis=2)
        return np.argmin(dists, axis=1)

## Use It — Real Data

### Experiment A: KNN on the Breast Cancer Wisconsin Dataset

First, a correctness check: does our from-scratch KNN produce the same predictions as scikit-learn's reference `KNeighborsClassifier`?

We load the dataset, split into train/test, and **scale the features** with `StandardScaler` — essential for any distance-based method, since KNN's distance calculation is otherwise dominated by whichever feature happens to have the largest raw numeric range.

In [4]:
data = load_breast_cancer()
X, y = data.data, data.target
print(f"Dataset shape: {X.shape[0]} examples, {X.shape[1]} features")
print(f"Classes: {data.target_names}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

Dataset shape: 569 examples, 30 features
Classes: ['malignant' 'benign']


Now we fit both the from-scratch and the scikit-learn KNN classifiers with `k=5`, and compare not just their accuracy but the **prediction agreement rate** — the fraction of test points where the two implementations output the exact same label.

In [5]:
my_knn = KNNFromScratch(k=5).fit(X_train_s, y_train)
my_pred = my_knn.predict(X_test_s)
my_acc = accuracy_score(y_test, my_pred)

sk_knn = KNeighborsClassifier(n_neighbors=5).fit(X_train_s, y_train)
sk_pred = sk_knn.predict(X_test_s)
sk_acc = accuracy_score(y_test, sk_pred)

agreement = np.mean(my_pred == sk_pred)
print(f"From-scratch KNN accuracy : {my_acc:.4f}")
print(f"sklearn KNN accuracy      : {sk_acc:.4f}")
print(f"Prediction agreement rate : {agreement:.4f}  (fraction of identical predictions)")

From-scratch KNN accuracy : 0.9591
sklearn KNN accuracy      : 0.9591
Prediction agreement rate : 1.0000  (fraction of identical predictions)


### Experiment B: Underfitting / Overfitting as a Function of K

Just like `max_depth` regularized decision trees in Chapter 1, **K** is the hyperparameter that controls KNN's bias/variance trade-off:

- **K = 1:** the model perfectly fits the training data (train accuracy = 1.0) but can overfit to noise
- **Large K:** predictions smooth out toward the majority class, risking underfitting

We sweep K across a wide range and track both training and test accuracy.

In [6]:
print(f"{'K':>4} | {'train acc':>10} | {'test acc':>9}")
print("-" * 30)
for k in [1, 2, 3, 5, 10, 20, 40, 80, 150]:
    knn = KNNFromScratch(k=k).fit(X_train_s, y_train)
    train_acc = accuracy_score(y_train, knn.predict(X_train_s))
    test_acc = accuracy_score(y_test, knn.predict(X_test_s))
    print(f"{k:>4} | {train_acc:>10.4f} | {test_acc:>9.4f}")

   K |  train acc |  test acc
------------------------------
   1 |     1.0000 |    0.9591
   2 |     0.9774 |    0.9591
   3 |     0.9724 |    0.9532
   5 |     0.9724 |    0.9591


  10 |     0.9698 |    0.9649
  20 |     0.9623 |    0.9415


  40 |     0.9523 |    0.9357


  80 |     0.9296 |    0.9415
 150 |     0.8995 |    0.9240


**Reading the table:** at `K=1`, training accuracy is perfect (each point is its own nearest neighbor) while test accuracy is lower — a classic overfitting signature. As `K` grows into the dozens, both curves tend to converge; if `K` grows too large, test accuracy can start to degrade again as the decision rule oversmooths toward the majority class.

### Experiment C: K-Means on the Wine Dataset

Now the unsupervised case. The **Wine** dataset has 3 known true classes, which we'll use *only for evaluation* (via the Adjusted Rand Index), never for training — K-Means itself never sees the labels.

We again scale the features, then compare our from-scratch K-Means against scikit-learn's `KMeans` (which additionally uses the smarter `k-means++` initialization and multiple restarts via `n_init`).

In [7]:
wine = load_wine()
Xw, yw = wine.data, wine.target
Xw_s = StandardScaler().fit_transform(Xw)
print(f"Dataset shape: {Xw.shape[0]} examples, {Xw.shape[1]} features, {len(set(yw))} true classes")

my_km = KMeansFromScratch(k=3, random_state=1).fit(Xw_s)
sk_km = KMeans(n_clusters=3, n_init=10, random_state=1).fit(Xw_s)

my_ari = adjusted_rand_score(yw, my_km.labels_)
sk_ari = adjusted_rand_score(yw, sk_km.labels_)

print(f"From-scratch K-Means Adjusted Rand Index vs true labels : {my_ari:.4f}")
print(f"sklearn K-Means      Adjusted Rand Index vs true labels : {sk_ari:.4f}")
print("(Adjusted Rand Index measures cluster/label agreement; 1.0 = perfect, 0.0 = random)")

Dataset shape: 178 examples, 13 features, 3 true classes
From-scratch K-Means Adjusted Rand Index vs true labels : 0.9149
sklearn K-Means      Adjusted Rand Index vs true labels : 0.8975
(Adjusted Rand Index measures cluster/label agreement; 1.0 = perfect, 0.0 = random)


### Experiment D: The Curse of Dimensionality

The book's claim (Section 2.5): as the number of dimensions grows, pairwise distances between random points **concentrate** — they all start to look roughly the same, even though their absolute magnitude grows.

To verify this, we generate random points uniformly in `[0, 1]^D` for increasing `D`, sample many pairwise distances, and track both the **mean** distance and the **relative spread** (`std / mean`).

In [8]:
print(f"{'Dimensions':>10} | {'mean dist':>10} | {'std dist':>9} | {'std/mean':>9}")
print("-" * 50)
for D in [2, 8, 32, 128, 512, 2048]:
    pts = RNG.uniform(0, 1, size=(200, D))
    idx_a = RNG.randint(0, 200, size=3000)
    idx_b = RNG.randint(0, 200, size=3000)
    d = np.linalg.norm(pts[idx_a] - pts[idx_b], axis=1)
    print(f"{D:>10} | {d.mean():>10.4f} | {d.std():>9.4f} | {d.std()/d.mean():>9.4f}")

Dimensions |  mean dist |  std dist |  std/mean
--------------------------------------------------
         2 |     0.5195 |    0.2513 |    0.4837
         8 |     1.1251 |    0.2483 |    0.2207
        32 |     2.2683 |    0.2751 |    0.1213
       128 |     4.5857 |    0.3792 |    0.0827


       512 |     9.1883 |    0.7165 |    0.0780


      2048 |    18.3439 |    1.5944 |    0.0869


**Reading the table:** as predicted in the book, the mean distance grows (roughly like $\sqrt{D}/3$ for points uniform in $[0,1]^D$), while the *relative* variance (`std/mean`) shrinks toward zero as `D` grows. In other words, in very high dimensions almost all pairs of points end up nearly the same distance apart — which is exactly why distance-based methods like KNN can struggle in high-dimensional, noisy feature spaces.

## Use It

| API / Function | When to use it |
|---|---|
| `KNNFromScratch(k).fit(X, y).predict(Xtest)` | Small-to-medium datasets, when you want an interpretable, non-parametric baseline classifier |
| `KMeansFromScratch(k).fit(X)` | Unsupervised grouping when you believe data forms roughly round, equally-sized clusters |
| `StandardScaler` (sklearn) | Always apply before KNN/K-Means when features are on different scales |
| `sklearn.neighbors.KNeighborsClassifier` | Production use — has KD-tree/ball-tree acceleration for large datasets |
| `sklearn.cluster.KMeans` | Production use — includes k-means++ smart initialization |

## Exercises

1. Modify `KNNFromScratch` to implement **weighted voting**, where closer neighbors get a vote of `exp(-0.5 * distance**2)` instead of an equal vote (Eq. 2.3 in the book).
2. Run Experiment B *without* `StandardScaler` and compare the accuracy curve — how much does feature scaling matter here?
3. Extend `KMeansFromScratch` to implement the **furthest-first heuristic** for initialization instead of random selection, and check whether the Adjusted Rand Index improves.

## Key Terms

| Term | Common Assumption | Precise Meaning |
|---|---|---|
| **Nearest Neighbor** | "The closest point is always right" | A model with zero training cost that defers all computation to prediction time, using distance as its only inductive bias |
| **Decision Boundary** | "Only matters for linear models" | The region in feature space where a classifier's predicted label changes; for 1-NN it is a jagged, Voronoi-like boundary |
| **Curse of Dimensionality** | "More features are always better" | The phenomenon where volumes, distances, and density estimates behave counter-intuitively as dimensionality grows, often making distance-based methods less reliable |
| **K-Means** | "It always finds the true clusters" | An iterative algorithm that provably converges to *a* local optimum of within-cluster squared distance, but not necessarily the global optimum or the "true" grouping |

## Summary

- Representing examples as points in space turns classification and clustering into geometric problems
- **KNN** classifies by majority vote among the `K` closest training points; it has no real training phase
- **K** is KNN's regularization knob: small `K` overfits, large `K` underfits
- **Feature scaling** is essential for any distance-based method
- **K-Means** alternates between assignment and mean re-estimation to find cluster centers, converging to a local (not necessarily global) optimum
- The **curse of dimensionality** causes distances to concentrate in high dimensions, weakening the signal that distance-based methods rely on

---

**Next:** Chapter 3 — Linear Models